# inference-mode-step composite — cx29: Adam state (v EMA) stays frozen under inference_mode

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `inference-mode-step`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "inference-mode-step"
DD_ATOM_IDS = ["ema-second-moment", "inference-mode-step"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "PyTorch: Inference mode step"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

An optimizer step is *defined* to mutate its state buffers (`v`, `m`, step counter) — that's how the EMAs evolve. But during **evaluation** you must NOT call `.step()` at all; the entire forward+backward pass should run under `t.inference_mode()` (or `t.no_grad()`), and `optimizer.step()` is simply not invoked. Result: `v` stays frozen.

The trap students hit: they wrap a *training* step in `inference_mode` (e.g. for a validation-style 'metrics' batch) and the leaf updates silently disappear — or, worse, they call `.step()` after a backward that didn't run, leaving `p.grad is None` and the step skipped without warning.

**The two atoms.**
- **ema-second-moment** — the `v` EMA update inside `.step()`.
- **inference-mode-step** — the eval-time discipline: `with t.inference_mode(): forward(x)` DOES NOT call `.step()`, so `v` is frozen.

**Anatomy of a `MiniRMSProp` with a `step(do_update: bool)` switch.**
```python
class MiniRMSProp:
    def __init__(self, params, lr, beta2, eps):
        self.params = list(params)
        self.v = [t.zeros_like(p) for p in self.params]
        self.lr, self.beta2, self.eps = lr, beta2, eps
        self.t = 0
    @t.inference_mode()
    def step(self):
        self.t += 1
        for p, v in zip(self.params, self.v):
            g = p.grad
            v.mul_(self.beta2).addcmul_(g, g, value=1-self.beta2)  # ema-second-moment.
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data.addcdiv_(g, v_hat.sqrt().add_(self.eps), value=-self.lr)
```

**Why care.** `model.eval()` flips Module-level `self.training`; `t.inference_mode()` flips the autograd switch. NEITHER touches the optimizer — that's on you. If you call `.step()` inside an eval loop, the optimizer happily updates `v` and `p` using whatever stale `p.grad` happens to be lying around.

### Composite Exercise — Adam state (v EMA) stays frozen under inference_mode

**Atoms exercised together**: `ema-second-moment`, `inference-mode-step`

Implement `cx29_make_mini_rmsprop()` — return the `MiniRMSProp` class.

Required structure:
- `__init__(self, params, lr=1e-2, beta2=0.999, eps=1e-8)`:
  - `self.params = list(params)`
  - `self.v = [t.zeros_like(p) for p in self.params]`
  - `self.lr, self.beta2, self.eps = lr, beta2, eps`
  - `self.t = 0`  # step counter.
- `step(self)` — runs under `t.inference_mode()`. For each `(p, v)`:
  - Increment `self.t` (once per call, before the loop).
  - `v.mul_(beta2).addcmul_(g, g, value=1-beta2)` (ema-second-moment).
  - `v_hat = v / (1 - beta2 ** self.t)`.
  - `p.data.addcdiv_(g, v_hat.sqrt().add_(eps), value=-lr)`.
- `zero_grad(self)` — sets each `p.grad` to `None` (mirrors `torch.optim.Optimizer.zero_grad(set_to_none=True)`).

The test runs two scenarios:
1. **Training scenario** — populate `p.grad`, call `.step()` repeatedly. Confirm `v` and `p` evolve.
2. **Eval scenario** — wrap a forward pass in `with t.inference_mode():` but DO NOT call `.step()`. Confirm `v` and `p` are byte-for-byte identical to before — Adam state is untouched.
Then it confirms `step()` mutates `v` in place (id and data_ptr unchanged) and that step `t` increments by exactly 1 per call.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx29_make_mini_rmsprop():
    """Return the MiniRMSProp class."""
    raise NotImplementedError

def _test_cx29():
    MiniRMSProp = cx29_make_mini_rmsprop()

    # Build a tiny 'model': just one Parameter.
    t.manual_seed(0)
    p = t.nn.Parameter(t.randn(3, 4))
    opt = MiniRMSProp([p], lr=1e-2, beta2=0.999, eps=1e-8)
    assert opt.t == 0
    assert len(opt.v) == 1 and opt.v[0].shape == p.shape
    assert t.allclose(opt.v[0], t.zeros_like(p))

    # Case A: training — .step() mutates p and v.
    p.grad = t.ones_like(p)
    v_id_before = id(opt.v[0])
    v_ptr_before = opt.v[0].data_ptr()
    p_id_before = id(p)
    p_ptr_before = p.data_ptr()
    p_before = p.detach().clone()
    opt.step()
    assert opt.t == 1, f'step counter must be 1 after one step; got {opt.t}'
    assert id(opt.v[0]) == v_id_before, 'v[0] was rebound'
    assert opt.v[0].data_ptr() == v_ptr_before, 'v[0] storage changed'
    assert id(p) == p_id_before and p.data_ptr() == p_ptr_before, 'p identity broken'
    assert not t.allclose(p.detach(), p_before), 'p should have moved after a training step'
    assert not t.allclose(opt.v[0], t.zeros_like(p)), 'v should be nonzero after step with grad=1'

    # Case B: EVAL — wrap a forward in inference_mode, do NOT call .step().
    v_snapshot = opt.v[0].clone()
    p_snapshot = p.detach().clone()
    t_snapshot = opt.t
    with t.inference_mode():
        # Simulate a forward pass — read p, compute some output. Do NOT call .step().
        out = p * 2.0
        _ = out.sum().item()
    assert t.allclose(opt.v[0], v_snapshot), 'eval scenario must not change v (no .step() called)'
    assert t.allclose(p.detach(), p_snapshot), 'eval scenario must not change p'
    assert opt.t == t_snapshot, 'eval scenario must not increment t'

    # Case C: zero_grad sets grads to None.
    opt.zero_grad()
    assert p.grad is None, 'zero_grad should set p.grad to None'

    # Case D: multi-step ema-second-moment trajectory matches the manual recurrence.
    t.manual_seed(7)
    p2 = t.nn.Parameter(t.randn(5))
    opt2 = MiniRMSProp([p2], lr=1e-2, beta2=0.9, eps=1e-8)
    v_ref = t.zeros(5)
    for step in range(1, 4):
        t.manual_seed(step * 11)
        g = t.randn(5)
        p2.grad = g.clone()
        opt2.step()
        v_ref = 0.9 * v_ref + 0.1 * g * g
    assert t.allclose(opt2.v[0], v_ref, atol=1e-7), 'v EMA trajectory disagrees with reference'
    assert opt2.t == 3, f'after 3 .step() calls, t should be 3; got {opt2.t}'
    _dd_passed.add('cx29')

_test_cx29()

<details><summary>Show solution — cx29</summary>

```python
def cx29_make_mini_rmsprop():
    class MiniRMSProp:
        def __init__(self, params, lr=1e-2, beta2=0.999, eps=1e-8):
            self.params = list(params)
            self.v = [t.zeros_like(p) for p in self.params]
            self.lr = lr
            self.beta2 = beta2
            self.eps = eps
            self.t = 0

        @t.inference_mode()
        def step(self):
            # Atom B (inference-mode-step): the decorator turns off autograd for the whole
            # body — required because we mutate leaf tensors in place.
            self.t += 1
            for p, v in zip(self.params, self.v):
                g = p.grad
                # Atom A (ema-second-moment): in-place v <- beta2*v + (1-beta2)*g*g.
                v.mul_(self.beta2).addcmul_(g, g, value=1 - self.beta2)
                v_hat = v / (1 - self.beta2 ** self.t)
                p.data.addcdiv_(g, v_hat.sqrt().add_(self.eps), value=-self.lr)

        def zero_grad(self):
            for p in self.params:
                p.grad = None

    return MiniRMSProp
```

Notice the test does NOT call `.step()` inside the eval `with t.inference_mode():` block — it's only the forward pass. That's the canonical pattern: the inference-mode decorator on `.step()` is for the LEAF MUTATIONS (so the in-place update on a `requires_grad=True` tensor is legal); the eval discipline (don't call `.step()` at all) is on the trainer loop. Both atoms have to be respected in their own scope.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["Optimizer: Adam EMA second moment", "PyTorch: Inference mode step"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()